# CV analyser: run the local-model eval on a Kaggle GPU

Runs the **same** evaluation as `scripts/run_eval.py` on a laptop, with the same
model (`qwen2.5:7b` through Ollama) and the same code from GitHub, but on Kaggle's
free GPU, so it takes minutes instead of over an hour.

**Before running:**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100).
2. *Settings → Internet*: **On** (needs a phone-verified Kaggle account).
3. *Add Input*: your private dataset made from `cv-eval-data.zip`
   (fictional eval CVs, answer key, job ads). Never upload real people's CVs.
4. *Run All*. Results are saved in `/kaggle/working/` (see *Output*).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install and start Ollama

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama --version

In [ ]:
import os
import subprocess
import time
import urllib.request

env = dict(os.environ, OLLAMA_KEEP_ALIVE="2h")
# The server runs for the whole session, so its log file stays open on purpose.
log = open("/kaggle/working/ollama.log", "w")  # noqa: SIM115
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is running")
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /kaggle/working/ollama.log")

In [ ]:
!ollama pull qwen2.5:7b 2>&1 | tail -n 2

## 3. Get the code

In [ ]:
!rm -rf /kaggle/working/cv-analyser-rag
!git clone -q --depth 1 https://github.com/shahriar62b/cv-analyser-rag.git /kaggle/working/cv-analyser-rag
%cd /kaggle/working/cv-analyser-rag
!git log --oneline -1
# Install the project without the laptop's exact pins, so Kaggle keeps its own
# GPU builds of PyTorch and NumPy.
!pip install -q -e . 2>&1 | tail -n 1

## 4. Find the uploaded data

In [ ]:
from pathlib import Path

key = next(Path("/kaggle/input").rglob("answer_key.json"))
cv_dir = next(p.parent for p in Path("/kaggle/input").rglob("cv_a_data_engineer.docx"))
jd_dir = next(p.parent for p in Path("/kaggle/input").rglob("jds_1.txt"))
print(key, cv_dir, jd_dir, sep="\n")

## 5. Run the eval

All 12 pairs in one go (Kaggle has enough memory, so no batches are needed).
`--keep-loaded` keeps the model in GPU memory between pairs.

In [ ]:
!LLM_PROVIDER=ollama python scripts/run_eval.py \
    --key "{key}" --cv-dir "{cv_dir}" --jd-dir "{jd_dir}" \
    --out /kaggle/working/kaggle_qwen7b.json --keep-loaded 2>&1 | grep -v -i "warning"

## 6. Check the model really ran on the GPU

In [ ]:
!ollama ps

## 7. Results

Download `kaggle_qwen7b.json` from the *Output* panel and put it in
`data/eval/results/` on the laptop to compare with `round1_7b.json`.

In [ ]:
import json
from pathlib import Path

summary = json.loads(Path("/kaggle/working/kaggle_qwen7b.json").read_text())["summary"]
print(
    json.dumps(
        {
            k: summary[k]
            for k in (
                "strict_accuracy",
                "lenient_accuracy",
                "false_met",
                "score_mean_abs_error",
                "negatives_passed",
                "extraction_coverage",
                "total_seconds",
            )
        },
        indent=2,
    )
)